# 🚀 Huấn Luyện Cham-DBNet (PaddleOCR Text Detection) Trên Lightning AI (GPU H100)

Pipeline huấn luyện chuyên sâu mô hình dò dòng chữ Chăm (**Cham-DBNet PP-OCRv4**):
- **Đặc trị dòng dính sát (Tight Line Spacing)**: Bổ sung 50% hard-cases cự ly dòng 1-5px.
- **Tối ưu hóa GPU H100 (80GB VRAM)**: Batch size lớn (64), FP16/AMP, đa luồng DataLoader.
- **1-Click Export**: Tự động đóng gói thành mô hình Inference nhẹ (~4.5MB) nén sẵn file ZIP tải về.

In [ ]:
# 1. Kiểm tra phần cứng GPU NVIDIA H100 và CUDA
!nvidia-smi

import torch
if torch.cuda.is_available():
    print(f"🔥 Đã phát hiện GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   Compute Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Chưa phát hiện GPU! Hãy chắc chắn bạn đã chọn H100 trong Lightning Studio.")

In [ ]:
# 2. Cài đặt PaddlePaddle GPU tương thích với CUDA 11.8 / 12.x
!pip install --upgrade pip
!pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install pyclipper shapely imgaug lmdb attrdict pillow opencv-python

import paddle
print(f"PaddlePaddle version: {paddle.__version__}")
print(f"CUDA compiled: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")
if paddle.is_compiled_with_cuda():
    print(f"Device Name: {paddle.device.cuda.get_device_name(0)}")

In [ ]:
# 3. Tải kho mã nguồn PaddleOCR (release/2.7)
import os
if not os.path.exists('PaddleOCR'):
    !git clone -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git

%cd PaddleOCR
!pip install -r requirements.txt
%cd ..

In [ ]:
# 4. Chuẩn bị Fonts chữ Chăm và Corpus văn bản
import os, urllib.request

os.makedirs('data/fonts', exist_ok=True)
os.makedirs('data/corpus', exist_ok=True)

# Tải bộ phông Google NotoSansCham chuẩn
font_urls = {
    'data/fonts/NotoSansCham-Regular.ttf': 'https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansCham/NotoSansCham-Regular.ttf',
    'data/fonts/NotoSansCham-Bold.ttf': 'https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansCham/NotoSansCham-Bold.ttf',
    'data/fonts/NotoSansCham-Black.ttf': 'https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansCham/NotoSansCham-Black.ttf'
}

for path, url in font_urls.items():
    if not os.path.exists(path):
        print(f"Tải phông: {path}...")
        urllib.request.urlretrieve(url, path)

print("✅ Bộ phông chữ Chăm đã sẵn sàng.")

In [ ]:
# 5. Sinh tập dữ liệu tổng hợp đa dòng đặc trị cự ly hẹp (Tight Line Spacing)
# Chạy kịch bản multiprocessing để sinh 15,000 trang train và 1,500 trang validation
!python3 scripts/generate_detector_data_v2.py --num_train 15000 --num_val 1500 --output_dir PaddleOCR/data/detector_v2 --workers 8

In [ ]:
# 6. Tải trọng số nền Pretrained PP-OCRv4 Detection
import os, urllib.request

os.makedirs('PaddleOCR/pretrain_models', exist_ok=True)
pretrain_url = 'https://paddleocr.bj.bcebos.com/pretrained/PPLCNetV3_x0_75_ocr_det.pdparams'
pretrain_path = 'PaddleOCR/pretrain_models/PPLCNetV3_x0_75_ocr_det.pdparams'
if not os.path.exists(pretrain_path):
    print(f"Đang tải pretrained weights PPLCNetV3...")
    urllib.request.urlretrieve(pretrain_url, pretrain_path)
    print("✅ Tải pretrained thành công!")
else:
    print("Pretrained model đã tồn tại.")

In [ ]:
# 7. Sao chép cấu hình tối ưu H100 vào PaddleOCR và Khởi chạy Huấn Luyện
import os, shutil

os.makedirs('PaddleOCR/configs/det', exist_ok=True)
shutil.copy('configs/det/ch_PP-OCRv4_det_h100.yml', 'PaddleOCR/configs/det/ch_PP-OCRv4_det_h100.yml')

%cd PaddleOCR
# Huấn luyện phân tán trên GPU H100 (batch 64, num_workers 8, 150 epochs)
!python3 -m paddle.distributed.launch --gpus '0' tools/train.py -c configs/det/ch_PP-OCRv4_det_h100.yml
%cd ..

In [ ]:
# 8. Xuất mô hình Inference (Inference Model Export)
import os

%cd PaddleOCR
best_model = 'output/ch_PP-OCRv4_det_cham_h100/best_accuracy'
if not os.path.exists(best_model + '.pdparams'):
    best_model = 'output/ch_PP-OCRv4_det_cham_h100/latest'

!python3 tools/export_model.py \
    -c configs/det/ch_PP-OCRv4_det_h100.yml \
    -o Global.pretrained_model={best_model} \
       Global.save_inference_dir=../output/ch_PP-OCRv4_det_cham_infer

%cd ..

In [ ]:
# 9. Đóng gói mô hình thành file cham_dbnet_v1_infer.zip để tải về
import zipfile, os

zip_path = 'cham_dbnet_v1_infer.zip'
target_dir = 'output/ch_PP-OCRv4_det_cham_infer'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(target_dir):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, os.path.dirname(target_dir))
            zipf.write(full_path, rel_path)

print(f"🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {zip_path}")
print(f"   Kích thước: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
print(f"👉 Bạn chỉ cần nhấp chuột phải vào file '{zip_path}' trên cây thư mục bên trái của Lightning AI và chọn 'Download'!")